## Phase 1: Setup & Load Data


In [ ]:
# ============================================================
# Marketing Customer Churn Prediction
# Phase 1: Import Libraries & Load Dataset
# ============================================================

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv("marketing_customer_churn.csv")

print("✅ Dataset loaded successfully!")


## Phase 2: Data Understanding (EDA)


In [ ]:
# ============================================================
# Dataset Overview
# ============================================================

print("Shape of the dataset:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nDataset Information:")
df.info()


In [ ]:
# ============================================================
# Missing Value Analysis
# ============================================================

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

missing_df = pd.DataFrame({
    "Missing Values": missing,
    "Percentage (%)": (missing / len(df) * 100).round(2)
})

display(missing_df)


In [ ]:
# ============================================================
# Duplicate Analysis
# ============================================================

full_duplicates = df.duplicated().sum()
customer_id_duplicates = df["Customer_ID"].duplicated().sum()

print(f"Full duplicate rows: {full_duplicates}")
print(f"Duplicate Customer_IDs: {customer_id_duplicates}")


In [ ]:
# ============================================================
# Target Variable Analysis
# ============================================================

print("Target Variable Distribution:\n")
print(df["Target_Churn"].value_counts())

print("\nPercentage Distribution:\n")
print((df["Target_Churn"].value_counts(normalize=True) * 100).round(2))

plt.figure(figsize=(5,4))
sns.countplot(data=df, x="Target_Churn")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.show()


In [ ]:
# ============================================================
# Categorical Variable Inspection
# ============================================================

categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    print(f"\n{'='*50}")
    print(f"{col}")
    print(f"{'='*50}")
    print(df[col].value_counts(dropna=False))


## Phase 3: Data Preparation


In [ ]:
# ============================================================
# Data Cleaning
# ============================================================

# Remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# Standardize text values
df["Gender"] = df["Gender"].str.strip().str.capitalize()
df["Email_Opt_In"] = df["Email_Opt_In"].str.strip().str.capitalize()

print("Dataset shape after cleaning:", df.shape)

print("\nGender:")
print(df["Gender"].value_counts())

print("\nEmail Opt-In:")
print(df["Email_Opt_In"].value_counts())


In [ ]:
# ============================================================
# Distribution of Numerical Columns with Missing Values
# ============================================================

missing_cols = [
    "Annual_Income",
    "Years_as_Customer",
    "App_Usage_Hours_Month",
    "Social_Media_Engagement_Score",
    "Discount_Usage_Rate",
    "Customer_Satisfaction_Score"
]

df[missing_cols].hist(figsize=(15, 8), bins=20)

plt.suptitle("Distribution of Numerical Variables with Missing Values", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Missing Value Imputation
# ============================================================

# Columns with missing values
missing_cols = [
    "Annual_Income",
    "Years_as_Customer",
    "App_Usage_Hours_Month",
    "Social_Media_Engagement_Score",
    "Discount_Usage_Rate",
    "Customer_Satisfaction_Score"
]

# Median imputation
for col in missing_cols:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)
    print(f"{col}: Missing values filled with median = {median_value:.2f}")

# Verify that no missing values remain
print("\nRemaining Missing Values:")
print(df[missing_cols].isnull().sum())


In [ ]:
# ============================================================
# Outlier Analysis
# ============================================================

outlier_cols = [
    "Annual_Income",
    "Total_Spend",
    "Avg_Order_Value"
]

plt.figure(figsize=(15,4))

for i, col in enumerate(outlier_cols, 1):
    plt.subplot(1, 3, i)
    sns.boxplot(y=df[col])
    plt.title(col)

plt.tight_layout()
plt.show()


### Outlier Treatment (IQR Capping)

The boxplots above show right-skewed distributions with high-end outliers in
`Annual_Income`, `Total_Spend`, and `Avg_Order_Value`. Instead of dropping these
rows (which would lose real high-value customers — important for a marketing/churn
model), we **cap (winsorize)** them at the IQR fences. This keeps the sample size
intact while limiting the influence of extreme values on distance-based and
linear models.


In [ ]:
# ============================================================
# Outlier Treatment - IQR Capping (Winsorization)
# ============================================================

def iqr_cap(series, k=1.5):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - k * IQR
    upper = Q3 + k * IQR
    return series.clip(lower=lower, upper=upper), lower, upper

outlier_cols = ["Annual_Income", "Total_Spend", "Avg_Order_Value"]

for col in outlier_cols:
    before_out = ((df[col] < df[col].quantile(0.25) - 1.5*(df[col].quantile(0.75)-df[col].quantile(0.25))) |
                  (df[col] > df[col].quantile(0.75) + 1.5*(df[col].quantile(0.75)-df[col].quantile(0.25)))).sum()
    df[col], lo, hi = iqr_cap(df[col])
    print(f"{col}: capped to [{lo:.2f}, {hi:.2f}]  |  {before_out} outlier rows treated")

# Verify with boxplots after treatment
plt.figure(figsize=(15,4))
for i, col in enumerate(outlier_cols, 1):
    plt.subplot(1, 3, i)
    sns.boxplot(y=df[col])
    plt.title(f"{col} (after capping)")
plt.tight_layout()
plt.show()


### Feature Engineering

We derive a few business-relevant features that are not directly present in the
raw data but are known to be strong churn signals in the marketing/CRM literature:

- **Campaign_Click_Rate** — engagement quality, not just volume of campaigns received
- **Spend_per_Purchase** — reframes `Total_Spend` relative to purchase frequency
- **Engagement_Index** — combines app usage and social media engagement into a single
  behavioral engagement score
- **Recency_Flag** — binary flag for customers inactive for a long period (a classic
  RFM/churn indicator)


In [ ]:
# ============================================================
# Feature Engineering
# ============================================================

df["Campaign_Click_Rate"] = (
    df["Num_Campaigns_Clicked"] / df["Num_Campaigns_Received"].replace(0, np.nan)
).fillna(0)

df["Spend_per_Purchase"] = (
    df["Total_Spend"] / df["Num_Purchases_Last_Year"].replace(0, np.nan)
).fillna(0)

df["Engagement_Index"] = (
    df["App_Usage_Hours_Month"].rank(pct=True) +
    df["Social_Media_Engagement_Score"].rank(pct=True)
) / 2

df["Recency_Flag"] = (df["Last_Purchase_Days_Ago"] > 90).astype(int)

print("New engineered features added:")
print(df[["Campaign_Click_Rate", "Spend_per_Purchase", "Engagement_Index", "Recency_Flag"]].describe())


### Feature Selection (Correlation Check)

Before modeling, we check pairwise correlation among numeric features to spot
redundancy. Highly correlated pairs (|r| > 0.85) can be dropped to reduce
multicollinearity, especially for Logistic Regression.


In [ ]:
# ============================================================
# Feature Selection - Correlation Heatmap
# ============================================================

numeric_df = df.select_dtypes(include=[np.number])

plt.figure(figsize=(14, 10))
corr = numeric_df.corr()
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Correlation Heatmap of Numeric Features")
plt.tight_layout()
plt.show()

# Flag highly correlated pairs
high_corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
)
high_corr_pairs.columns = ["Feature 1", "Feature 2", "Correlation"]
high_corr_pairs = high_corr_pairs[high_corr_pairs["Correlation"].abs() > 0.85]

print("Highly correlated feature pairs (|r| > 0.85):")
display(high_corr_pairs if len(high_corr_pairs) else "None found - no redundant features to drop.")


In [ ]:
# ============================================================
# Feature Encoding
# ============================================================

# Create a copy for modeling
df_model = df.copy()

# Remove Customer_ID
df_model.drop(columns=["Customer_ID"], inplace=True)

# Encode target variable
df_model["Target_Churn"] = df_model["Target_Churn"].map({
    "No": 0,
    "Yes": 1
})

# Identify categorical predictor columns
categorical_cols = df_model.select_dtypes(include="object").columns.tolist()

# One-Hot Encode predictors
df_model = pd.get_dummies(
    df_model,
    columns=categorical_cols,
    drop_first=True
)

print("Encoded dataset shape:", df_model.shape)

display(df_model.head())


In [ ]:
# ============================================================
# Train-Test Split
# ============================================================

from sklearn.model_selection import train_test_split

# Features and target
X = df_model.drop("Target_Churn", axis=1)
y = df_model["Target_Churn"]

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set :", X_test.shape)

print("\nTarget Distribution")
print("Training:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting:")
print(y_test.value_counts(normalize=True).round(3))


## Phase 4: Model Building & Hyperparameter Tuning


In [ ]:
# ============================================================
# Import Machine Learning Libraries
# ============================================================

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)


In [ ]:
# ============================================================
# Model Evaluation Function
# ============================================================

import time

# Store model results
results = []

def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):

    # Train model
    start = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start

    # Predictions
    y_pred = model.predict(X_test)

    # Probabilities (needed for ROC-AUC)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = None

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    roc_auc = roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan

    # Save results
    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc,
        "Training Time (s)": training_time
    })

    # Print metrics
    print("=" * 50)
    print(model_name)
    print("=" * 50)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"ROC-AUC  : {roc_auc:.4f}")
    print(f"Training Time: {training_time:.4f} sec")

    # Classification Report
    print("\nClassification Report")
    print(classification_report(y_test, y_pred))

    # Confusion Matrix
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
    plt.title(model_name)
    plt.show()


### Logistic Regression (with hyperparameter tuning)

Scaling is required here since Logistic Regression is a distance/gradient-based
linear model sensitive to feature magnitude. Tree-based models below do not
require scaling, since they split on thresholds rather than distances.


In [ ]:
# ============================================================
# Logistic Regression - GridSearchCV
# ============================================================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l2"],
    "solver": ["lbfgs"]
}

lr_grid = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=1000),
    lr_param_grid, cv=5, scoring="f1", n_jobs=-1
)
lr_grid.fit(X_train_scaled, y_train)
print("Best Logistic Regression params:", lr_grid.best_params_)

lr = evaluate_model(
    model=lr_grid.best_estimator_,
    X_train=X_train_scaled, X_test=X_test_scaled,
    y_train=y_train, y_test=y_test,
    model_name="Logistic Regression (Tuned)"
)


In [ ]:
# ============================================================
# Decision Tree - GridSearchCV
# ============================================================

dt_param_grid = {
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5]
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_param_grid, cv=5, scoring="f1", n_jobs=-1
)
dt_grid.fit(X_train, y_train)
print("Best Decision Tree params:", dt_grid.best_params_)

dt = evaluate_model(
    model=dt_grid.best_estimator_,
    X_train=X_train, X_test=X_test,
    y_train=y_train, y_test=y_test,
    model_name="Decision Tree (Tuned)"
)


In [ ]:
# ============================================================
# Random Forest - GridSearchCV
# ============================================================

rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_leaf": [1, 2, 5]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid, cv=5, scoring="f1", n_jobs=-1
)
rf_grid.fit(X_train, y_train)
print("Best Random Forest params:", rf_grid.best_params_)

rf = evaluate_model(
    model=rf_grid.best_estimator_,
    X_train=X_train, X_test=X_test,
    y_train=y_train, y_test=y_test,
    model_name="Random Forest (Tuned)"
)


In [ ]:
# ============================================================
# XGBoost - GridSearchCV
# ============================================================

xgb_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2]
}

xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric="logloss"),
    xgb_param_grid, cv=5, scoring="f1", n_jobs=-1
)
xgb_grid.fit(X_train, y_train)
print("Best XGBoost params:", xgb_grid.best_params_)

xgb = evaluate_model(
    model=xgb_grid.best_estimator_,
    X_train=X_train, X_test=X_test,
    y_train=y_train, y_test=y_test,
    model_name="XGBoost (Tuned)"
)


### Feature Importance (Random Forest)

To interpret which variables the tuned Random Forest relies on most, we extract
`feature_importances_` from the best estimator found by GridSearchCV.

In [ ]:
# ============================================================
# Feature Importance - Random Forest (Tuned)
# ============================================================

importances = rf_grid.best_estimator_.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)

top_n = 12
plt.figure(figsize=(9, 6))
sns.barplot(x=feat_imp.head(top_n).values, y=feat_imp.head(top_n).index, color="steelblue")
plt.title(f"Top {top_n} Feature Importances - Random Forest (Tuned)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print(feat_imp.head(top_n))

## Phase 5: Model Comparison Leaderboard


In [ ]:
# ============================================================
# Model Comparison
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)

display(results_df)


In [ ]:
# ============================================================
# Final Performance Comparison
# ============================================================

import matplotlib.pyplot as plt

metrics = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]

comparison = results_df.set_index("Model")[metrics]

comparison.plot(kind="bar", figsize=(10, 6))



plt.title("Performance Comparison of Classification Models", fontsize=14)
plt.xlabel("Models")
plt.ylabel("Score")
plt.ylim(0,1)
plt.xticks(rotation=0)

plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.legend(
    title="Metrics",
    bbox_to_anchor=(1.02,1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


### Business Interpretation of Results

- **Recall on the churn class (1) matters more than overall accuracy.** Missing an
  actual churner (a false negative) means losing that customer with no retention
  intervention, which is far costlier than a false alarm (offering a retention
  discount to a customer who wasn't going to churn anyway).
- **ROC-AUC** gives a threshold-independent view of how well each model separates
  churners from non-churners, useful since the marketing team may want to adjust
  the decision threshold (e.g., target the top 20% highest-risk customers for a
  retention campaign rather than using a fixed 0.5 cutoff).
- The **leaderboard's top model** should be selected primarily on **F1-score**
  (balances precision/recall) and **Recall**, not accuracy alone, since the churn
  classes are imbalanced (~65% stay / 35% churn) and the business cost of a missed
  churner outweighs the cost of a wasted retention offer.
- Feature importance from the best tree-based model (Random Forest / XGBoost)
  should be reviewed to identify **actionable levers** for the marketing team —
  e.g., if `Customer_Satisfaction_Score` and `Discount_Usage_Rate` rank highest,
  campaigns should focus on satisfaction recovery and targeted discounting for
  at-risk segments.


## Democratizing the Solution

To make this model usable by non-technical marketing stakeholders rather than
staying a notebook-only artifact, we propose:

1. **Streamlit web app** — a simple form where a marketing analyst enters/uploads
   customer attributes and instantly gets a churn probability and risk tier
   (Low/Medium/High), with the model and preprocessing pipeline serialized via
   `joblib` and loaded at app startup.
2. **Batch scoring pipeline** — a scheduled script that scores the entire active
   customer base weekly and pushes a ranked "at-risk customer" list into the CRM
   (e.g., Salesforce/HubSpot) so retention campaigns can be triggered automatically.
3. **Power BI / Tableau dashboard** — surfaces the leaderboard, churn drivers
   (feature importance), and a live churn-risk distribution for business
   stakeholders who won't touch Python or the notebook directly.
4. **REST API (FastAPI/Flask)** — wraps the trained model behind an endpoint so it
   can be integrated into existing marketing automation tools.

This ensures the analytics output moves from a one-off assignment notebook to a
recurring, actionable business tool.
